In [1]:
%load_ext autoreload
%autoreload 2

import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
import math
from scipy.io import loadmat

from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference

E0000 00:00:1771446401.287910 2787779 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771446401.293123 2787779 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# les donnees caffeNet GoogleNet

In [2]:
featuresToUse = ["CaffeNet4096", "GoogleNet1024"] 
#featuresToUse = ["CaffeNet4096", "CaffeNet4096"] 
sourceDomainName = ['caltech10'] #['caltech10','amazon','webcam']
targetDomainName = ['caltech10'] #['caltech10','amazon','webcam']

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[0],
                                                 "caltech10" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[1],
                                                 "amazon" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1


In [3]:
results = []
numRepetitions = 1
alpha =0# hyperparamètre devant la loss a été optimé
#prop_target_values_s = [0.01, 0.05, 0.1, 0.2, 0.4]
#prop_target_values_p = [0.01, 0.05, 0.1, 0.2, 0.4]



# Je compare COOT et JDCOOT dans le cas non supervisé (on observe tous les labels dans source et aucun dans target)
# et dans le cas où l'outcome est discret
# Il faut regarder l'erreur sur les données target qui ont servi à l'apprentissage: pure_target et l'erreur sur les données target test : test_target
# on utilise les fonctions  discrete_unsupervised_coot et  discrete_unsupervised_jdcoot
# Elles sont rangées dans src/models.
# Malheureusement les hyperparamètres sont à changer dans les fonctions mais je pense que c'est tres visible pour toi

In [4]:
for repe in range(numRepetitions):
    print("num repe :", repe + 1)

    a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
    b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

    S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
    T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
    S = source.iloc[a, :].reset_index(drop=True)
    T = target.iloc[b, :].reset_index(drop=True)

    # =========================================================
    # UNSUPERVISED
    # =========================================================
     # COOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_coot(S, T, S_test, T_test,alpha=alpha)

    results.append({
        "repetition": repe,
        "recoding": "coot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })
    
    # JDCOOT
    pure_source, pure_target, test_source, test_target = \
        discrete_unsupervised_jdcoot(S, T, S_test, T_test,alpha=alpha)

    results.append({
        "repetition": repe,
        "recoding": "jdcoot",
        "learning": "unsupervised",
        "prop_source": 1,
        "prop_target": 0,
        "pure_source": pure_source,
        "test_source": test_source,
        "pure_target": pure_target,
        "test_target": test_target,
    })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "prop_source", "prop_target"],
        as_index=False
    )
    .agg(
        pure_source_mean=("pure_source", "mean"),
        pure_source_var=("pure_source", "var"),
        test_source_mean=("test_source", "mean"),
        test_source_var=("test_source", "var"),
        pure_target_mean=("pure_target", "mean"),
        pure_target_var=("pure_target", "var"),
        test_target_mean=("test_target", "mean"),
        test_target_var=("test_target", "var"),
    )
)

df_summary
  

num repe : 1
Delta:       0.0157827 	 Loss:       2.1848293
Delta:       0.0232951 	 Loss:       1.8677497
Delta:       0.0207459 	 Loss:       1.6143343
Delta:       0.0185111 	 Loss:       1.5256085
Delta:       0.0151492 	 Loss:       1.4850681
Delta:       0.0116542 	 Loss:       1.4722500
Delta:       0.0092630 	 Loss:       1.4668542
Delta:       0.0081522 	 Loss:       1.4634168
Delta:       0.0070847 	 Loss:       1.4609917
Delta:       0.0051739 	 Loss:       1.4600602
Delta:       0.0043654 	 Loss:       1.4597390
Delta:       0.0040593 	 Loss:       1.4594256
Delta:       0.0031776 	 Loss:       1.4591426
Delta:       0.0019228 	 Loss:       1.4590228
Delta:       0.0010763 	 Loss:       1.4590110
Delta:       0.0029452 	 Loss:       1.4589568
Delta:       0.0025422 	 Loss:       1.4587679
Delta:       0.0023843 	 Loss:       1.4586903
Delta:       0.0005247 	 Loss:       1.4586835
Delta:       0.0000453 	 Loss:       1.4586660
Delta:       0.0000000 	 Loss:       1.4586660


I0000 00:00:1771446482.525454 2787779 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31129 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1771446483.592462 2788103 service.cc:148] XLA service 0x7a85b0e49450 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771446483.592507 2788103 service.cc:156]   StreamExecutor device (0): Tesla V100S-PCIE-32GB, Compute Capability 7.0
I0000 00:00:1771446483.613203 2788103 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1771446483.709445 2788103 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Delta: 0.023295114191058237 	  Loss: 1.8677497390798656 	 Accuracy: 0.23468057366362452
Delta: 0.020745860362772036 	  Loss: 1.614334281548953 	 Accuracy: 0.3455019556714472
Delta: 0.018511068563549577 	  Loss: 1.5256084768397276 	 Accuracy: 0.3624511082138201
Delta: 0.015149189157070036 	  Loss: 1.4850680805794934 	 Accuracy: 0.3455019556714472
Delta: 0.011654204977274662 	  Loss: 1.4722500172281066 	 Accuracy: 0.33376792698826596
Delta: 0.009262967958249636 	  Loss: 1.4668542071597872 	 Accuracy: 0.32073011734028684
Delta: 0.008152179017349853 	  Loss: 1.4634168471632614 	 Accuracy: 0.318122555410691
Delta: 0.007084727927226827 	  Loss: 1.460991669608759 	 Accuracy: 0.3246414602346806
Delta: 0.005173922722214219 	  Loss: 1.4600601832974234 	 Accuracy: 0.3259452411994785
Delta: 0.00436539133824841 	  Loss: 1.4597390424452859 	 Accuracy: 0.3285528031290743
Delta: 0.004059262825002337 	  Loss: 1.4594256463178783 	 Accuracy: 0.3272490221642764
Delta: 0.00317760161761542 	  Loss: 1.459142

,recoding,learning,prop_source,prop_target,pure_source_mean,pure_source_var,test_source_mean,test_source_var,pure_target_mean,pure_target_var,test_target_mean,test_target_var
0,coot,unsupervised,1,0,1.0,NaN,1.0,NaN,0.323338,NaN,0.335079,NaN
1,jdcoot,unsupervised,1,0,1.0,NaN,1.0,NaN,0.322034,NaN,0.324607,NaN


In [12]:
perf_test_target

np.float64(0.9322916666666666)

In [5]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)

import numpy as np
from sklearn.preprocessing import OneHotEncoder as onehot
from sklearn.model_selection import train_test_split
from jdcoot.utils import xcolumns, discrete_classifiers, discrete_accuracy
prop_source = 0.8
prop_target = 0.8
source_levels = np.sort(np.unique(S.Z))
target_levels = np.sort(np.unique(T.Z))

nClass = len(np.union1d(source_levels, target_levels))
categories = [np.arange(nClass)]

enc = onehot(handle_unknown="ignore", sparse_output=False, categories=categories)



x_source_train = S.loc[:, xcolumns(S)].values
z_source_train = enc.fit_transform(S.Z.values[:, np.newaxis])

x_target_train = T.loc[:, xcolumns(T)].values
z_target_train = enc.fit_transform(T.Z.values[:, np.newaxis])

x_source_test = S_test.loc[:, xcolumns(S_test)].values
z_source_test = S_test.Z.values

x_target_test = T_test.loc[:, xcolumns(T_test)].values
z_target_test = T_test.Z.values

clf_source, clf_target = discrete_classifiers(source, target, "relu", "softmax")

clf_target.fit(x_target_train, z_target_train, batch_size=20, epochs=20, verbose=0)
clf_source.fit(x_source_train, z_source_train, batch_size=20, epochs=20, verbose=0)

z_target_pred = enc.inverse_transform(
        clf_target.predict(x_target_test, verbose=0)
    ).ravel()
z_source_pred = enc.inverse_transform(
        clf_source.predict(x_source_test, verbose=0)
    ).ravel()

perf_test_source = discrete_accuracy(z_source_pred,S_test.Z)
perf_test_target = discrete_accuracy(z_target_pred, T_test.Z)

In [6]:
perf_test_target

np.float64(0.9581151832460733)

In [ ]:
import numpy as np
# Exemple de valeurs à tester pour alpha
alpha_values = np.linspace(5, 10, 1)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique

for a in alpha_values:

    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(source, target, source, target, alpha=a)

    score = test_target  
    
    if score > best_score:
        best_score = score
        best_alpha = a

print("Meilleur alpha :", best_alpha)
print("Score associé :", best_score)

Delta: 0.02285994509023098 	  Loss: 1.869471086817049 	 Accuracy: 0.2954070981210856
Delta: 0.020755398634391197 	  Loss: 1.572885502456149 	 Accuracy: 0.44780793319415446
Delta: 0.01594605661987325 	  Loss: 1.513360731901929 	 Accuracy: 0.4707724425887265
Delta: 0.012886153109281115 	  Loss: 1.4956809266314577 	 Accuracy: 0.4822546972860125
Delta: 0.01155058693221235 	  Loss: 1.4834797745252208 	 Accuracy: 0.5031315240083507
Delta: 0.010334778346365129 	  Loss: 1.4756083630514985 	 Accuracy: 0.5125260960334029
Delta: 0.009244732479946323 	  Loss: 1.470911048511295 	 Accuracy: 0.5219206680584552
Delta: 0.007803190803917955 	  Loss: 1.4684483607623302 	 Accuracy: 0.5302713987473904
Delta: 0.005898475503820892 	  Loss: 1.467418967761617 	 Accuracy: 0.5292275574112735
Delta: 0.005203638415890617 	  Loss: 1.4667193204962785 	 Accuracy: 0.5313152400835073
Delta: 0.0037644842546464205 	  Loss: 1.4663015478013763 	 Accuracy: 0.5302713987473904
Delta: 0.0021168269171750386 	  Loss: 1.466157778

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.00546544603611695 	  Loss: 1.4512452441981236 	 Accuracy: 0.4394572025052192
Delta: 0.005035983649294466 	  Loss: 1.4504464807616109 	 Accuracy: 0.44154488517745305
Delta: 0.004379403005864963 	  Loss: 1.4497765788389718 	 Accuracy: 0.4394572025052192
Delta: 0.0046503755743603345 	  Loss: 1.4494044114731113 	 Accuracy: 0.44572025052192066
Delta: 0.005044633171470715 	  Loss: 1.4488798065472241 	 Accuracy: 0.44467640918580376
Delta: 0.0043300498713281545 	  Loss: 1.4483773718638338 	 Accuracy: 0.44572025052192066
Delta: 0.0037714301343850535 	  Loss: 1.448085873598652 	 Accuracy: 0.44467640918580376
Delta: 0.003763145160200546 	  Loss: 1.447882014475737 	 Accuracy: 0.44572025052192066
Delta: 0.003408841206970317 	  Loss: 1.4477188822293956 	 Accuracy: 0.44363256784968685
Delta: 0.0017285748752013164 	  Loss: 1.447603112468763 	 Accuracy: 0.44363256784968685
Delta: 0.0027585818602206026 	  Loss: 1.447647445058173 	 Accuracy: 0.44154488517745305
Delta: 0.0026609163832794548 	  Lo

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.010303362549233768 	  Loss: 1.4812953742567265 	 Accuracy: 0.4081419624217119
Delta: 0.007395851742429948 	  Loss: 1.478235355809598 	 Accuracy: 0.4112734864300626
Delta: 0.006093113128687175 	  Loss: 1.4767154902838748 	 Accuracy: 0.4133611691022965
Delta: 0.004184949310856409 	  Loss: 1.4760803894779633 	 Accuracy: 0.4112734864300626
Delta: 0.003731818124716924 	  Loss: 1.4757746971471661 	 Accuracy: 0.4112734864300626
Delta: 0.0026298670969003074 	  Loss: 1.4756016539623815 	 Accuracy: 0.4123173277661795
Delta: 0.0020439228287730217 	  Loss: 1.4758048382796842 	 Accuracy: 0.4112734864300626
Delta: 0.0017787476889054765 	  Loss: 1.4757800745598426 	 Accuracy: 0.4112734864300626
Delta: 0.0012042936416009613 	  Loss: 1.4758253731352755 	 Accuracy: 0.4123173277661795
Delta: 0.00057958161846023 	  Loss: 1.4757471675207512 	 Accuracy: 0.4091858037578288
Delta: 0.0024970895920456387 	  Loss: 1.47578712948445 	 Accuracy: 0.4123173277661795
Delta: 0.0018621433812184135 	  Loss: 1.47

...

Delta: 0.01534174665274625 	  Loss: 1.5276382504487336 	 Accuracy: 0.3872651356993737
Delta: 0.010823796962774208 	  Loss: 1.5191241072933848 	 Accuracy: 0.3883089770354906
Delta: 0.008653769949171657 	  Loss: 1.516348751578179 	 Accuracy: 0.3883089770354906
Delta: 0.006548773148274266 	  Loss: 1.5150563591849455 	 Accuracy: 0.3893528183716075
Delta: 0.005204747788440986 	  Loss: 1.514161908399628 	 Accuracy: 0.3893528183716075
Delta: 0.004440117596930534 	  Loss: 1.5132335614610635 	 Accuracy: 0.3883089770354906
Delta: 0.004255412657746312 	  Loss: 1.5126019596464484 	 Accuracy: 0.3893528183716075
Delta: 0.0039126109984646655 	  Loss: 1.5120707636782953 	 Accuracy: 0.3893528183716075
Delta: 0.0033358380405259895 	  Loss: 1.51265483196605 	 Accuracy: 0.3903966597077244
Delta: 0.0015094822047320674 	  Loss: 1.512449867928398 	 Accuracy: 0.3893528183716075
Delta: 0.0021376167083853545 	  Loss: 1.5128915802676803 	 Accuracy: 0.3914405010438413
Delta: 0.001857263985177184 	  Loss: 1.512904

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.014549738477658301 	  Loss: 1.569536334042004 	 Accuracy: 0.34237995824634654
Delta: 0.00998736736534569 	  Loss: 1.5621043780767687 	 Accuracy: 0.34237995824634654
Delta: 0.0074055189708606296 	  Loss: 1.5602863762701538 	 Accuracy: 0.34237995824634654
Delta: 0.0062984547357156275 	  Loss: 1.55960678240498 	 Accuracy: 0.34237995824634654
Delta: 0.0047525128053203134 	  Loss: 1.5591002068114372 	 Accuracy: 0.34237995824634654
Delta: 0.003312235645070973 	  Loss: 1.5588777318734026 	 Accuracy: 0.34133611691022964
Delta: 0.003238708900811573 	  Loss: 1.5587792273294783 	 Accuracy: 0.34133611691022964
Delta: 0.003142481253012692 	  Loss: 1.559136768191038 	 Accuracy: 0.3444676409185804
Delta: 0.002998516436367603 	  Loss: 1.5584015664325874 	 Accuracy: 0.34237995824634654
Delta: 0.0024078453604739277 	  Loss: 1.5583636228614481 	 Accuracy: 0.34029227557411273
Delta: 0.0028631287962335014 	  Loss: 1.557828315702741 	 Accuracy: 0.3444676409185804
Delta: 0.003759131202708631 	  Loss

/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.014768784518280773 	  Loss: 1.551728710102368 	 Accuracy: 0.36221294363256784
Delta: 0.010471849931716477 	  Loss: 1.5451188613491298 	 Accuracy: 0.36325678496868474
Delta: 0.008344095054809654 	  Loss: 1.542504041701989 	 Accuracy: 0.36325678496868474
Delta: 0.006607417883653232 	  Loss: 1.5411794122727867 	 Accuracy: 0.36430062630480164
Delta: 0.006608169903030863 	  Loss: 1.539472208215938 	 Accuracy: 0.36430062630480164
Delta: 0.005660913308214389 	  Loss: 1.5393429152153908 	 Accuracy: 0.36325678496868474
Delta: 0.004939873864380276 	  Loss: 1.53879340928262 	 Accuracy: 0.36325678496868474
Delta: 0.003650651797461512 	  Loss: 1.5384176472705429 	 Accuracy: 0.36221294363256784
Delta: 0.003038859318668768 	  Loss: 1.538392655412396 	 Accuracy: 0.36221294363256784
Delta: 0.0006466833175214422 	  Loss: 1.5381867609969042 	 Accuracy: 0.36221294363256784
Delta: 3.078016622117259e-05 	  Loss: 1.5381877625857316 	 Accuracy: 0.35908141962421714
Delta: 0.0018078262859519392 	  Loss

In [ ]:
 best_alpha